# Part 3: SQL Analysis

## Goal
Use SQL on the cleaned Global Findex dataset to answer business-style question in  a reproducible way.

## What this notebook will do
- Load the cleaned dataset  into SQLITE
- Validate the SQL table
- Analyze overall coverage
- Start Country-level analysis with Nepal

## Imports

In [1]:
import sqlite3
from pathlib import Path 
import pandas as pd

In [4]:
project_path = Path(r"D:\financial-inclusion-gap-analysis")
csv_path = project_path / "data" / "processed" / "findex_analysis_ready.csv"
db_path = project_path / "data" / "processed" / "findex_analysis.db"

print(csv_path)
print(db_path)
print(csv_path.exists(), db_path.exists())

D:\financial-inclusion-gap-analysis\data\processed\findex_analysis_ready.csv
D:\financial-inclusion-gap-analysis\data\processed\findex_analysis.db
True False


## Load the cleaned dataset into SQLite
We use SQLite so we can practice SQl on the processed dataset without depending on external database software

In [6]:
df = pd.read_csv(csv_path)
df.shape

(8577, 18)

In [7]:
conn = sqlite3.connect(db_path)
df.to_sql("findex_clean", conn, if_exists="replace", index=False)
print("Table loaded to SQLite")

Table loaded to SQLite


## First SQL Checks
These checks confirms that the SQL tabel matches the cleaned dataset.

In [8]:
query = "SELECT COUNT(*) AS total_rows FROM findex_clean"
pd.read_sql_query(query, conn)

,total_rows
0,8577


In [14]:
query = """
SELECT entity_type, COUNT(*) AS row_count
FROM findex_clean
GROUP BY entity_type
ORDER BY row_count DESC;
"""
pd.read_sql(query, conn)

,entity_type,row_count
0,Country,7893
1,Aggregate,684


In [16]:
query = """
SELECT year, COUNT(*) AS row_count
FROM findex_clean
GROUP BY year
ORDER BY year
"""
pd.read_sql(query, conn)

,year,row_count
0,2011,1516
1,2014,1686
2,2017,1734
3,2021,1483
4,2022,176
5,2024,1982


In [17]:
query = """
SELECT COUNT(DISTINCT country_code) AS distinct_country_codes
FROM findex_clean
"""
pd.read_sql(query, conn)

,distinct_country_codes
0,174


## Nepal Analysis
We now filter to Nepal and inspect financial inclusion indicators over time.

In [18]:
query = """
SELECT *
FROM findex_clean
WHERE country_code = 'NPL'
LIMIT 10
"""
pd.read_sql(query, conn)

,country,country_code,entity_type,year,survey_period_type,adult_population,region,income_group,group,group2,account_ownership,financial_institution_account,mobile_money_account,digital_access,digital_merchant_payment,digital_payment_made,digital_payment_received,inactive_account
0,Nepal,NPL,Country,2011,Standard survey wave,17498633.0,South Asia (excluding high income),Lower middle income,all,all,0.253086,0.253086,NaN,NaN,NaN,NaN,NaN,NaN
1,Nepal,NPL,Country,2014,Standard survey wave,18161615.0,South Asia (excluding high income),Lower middle income,all,all,0.338014,0.338014,0.003360,NaN,NaN,0.056214,0.065320,0.057346
2,Nepal,NPL,Country,2017,Standard survey wave,18869412.0,South Asia (excluding high income),Lower middle income,all,all,0.453856,0.453856,NaN,NaN,NaN,0.091124,0.110819,0.112355
3,Nepal,NPL,Country,2021,Standard survey wave,20254590.0,South Asia (excluding high income),Lower middle income,all,all,0.540027,0.528021,0.060919,NaN,0.054149,0.187359,0.194268,0.140585
4,Nepal,NPL,Country,2024,Standard survey wave,21168213.0,South Asia (excluding high income),Lower middle income,all,all,0.599911,0.575257,0.105834,0.176216,0.092866,0.170105,0.212262,0.106261
5,Nepal,NPL,Country,2011,Standard survey wave,17498633.0,South Asia (excluding high income),Lower middle income,gender,men,0.295582,0.295582,NaN,NaN,NaN,NaN,NaN,NaN
6,Nepal,NPL,Country,2011,Standard survey wave,17498633.0,South Asia (excluding high income),Lower middle income,gender,women,0.212196,0.212196,NaN,NaN,NaN,NaN,NaN,NaN
7,Nepal,NPL,Country,2014,Standard survey wave,18161615.0,South Asia (excluding high income),Lower middle income,gender,men,0.367245,0.367245,NaN,NaN,NaN,NaN,NaN,NaN
8,Nepal,NPL,Country,2014,Standard survey wave,18161615.0,South Asia (excluding high income),Lower middle income,gender,women,0.312665,0.312665,NaN,NaN,NaN,NaN,NaN,NaN
9,Nepal,NPL,Country,2017,Standard survey wave,18869412.0,South Asia (excluding high income),Lower middle income,gender,men,0.499813,0.499813,NaN,NaN,NaN,NaN,0.142650,0.136624


In [21]:
query = """
SELECT
    year,
    account_ownership,
    digital_access,
    mobile_money_account,
    inactive_account
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" = 'all'
    AND "group2" = 'all'
ORDER BY year
"""
pd.read_sql(query, conn)

,year,account_ownership,digital_access,mobile_money_account,inactive_account
0,2011,0.253086,NaN,NaN,NaN
1,2014,0.338014,NaN,0.003360,0.057346
2,2017,0.453856,NaN,NaN,0.112355
3,2021,0.540027,NaN,0.060919,0.140585
4,2024,0.599911,0.176216,0.105834,0.106261


## Summary

Initial SQL validation confirms that the cleaned dataset was loaded correctly into SQLite.

The first Nepal-focused query filters to:
- `country_code = 'NPL'`
- `entity_type = 'Country'`
- `"group" = 'all'`
- `"group2" = 'all'`

This gives the national overall view for Nepal across survey years and avoids mixing subgroup rows into the trend.

## Nepal Trend Analysis
We now focus on specific indicators over time so the sql analysis starts producing interpretable country insights.

In [24]:
query = """
SELECT
    year,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" = 'all'
    AND "group2" = 'all'
"""
pd.read_sql(query, conn)

,year,account_ownership_pct
0,2011,25.31
1,2014,33.80
2,2017,45.39
3,2021,54.00
4,2024,59.99


In [25]:
query = """
SELECT
    year,
    ROUND(digital_access * 100, 2) AS account_ownership_pct
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" = 'all'
    AND "group2" = 'all'
"""
pd.read_sql(query, conn)

,year,account_ownership_pct
0,2011,NaN
1,2014,NaN
2,2017,NaN
3,2021,NaN
4,2024,17.62


In [26]:
query = """
SELECT
    year,
    ROUND(mobile_money_account * 100, 2) AS mobile_money_account_pct,
    ROUND(inactive_account * 100, 2) AS inactive_account_pct
FROM findex_clean
WHERE country_code = 'NPL'
    AND entity_type = 'Country'
    AND "group" = 'all'
    AND "group2" = 'all'
ORDER BY year
"""
pd.read_sql(query, conn)

,year,mobile_money_account_pct,inactive_account_pct
0,2011,NaN,NaN
1,2014,0.34,5.73
2,2017,NaN,11.24
3,2021,6.09,14.06
4,2024,10.58,10.63


## Benchmark Analysis
The next step is to compare Nepal with broader benchmarks so we can judge whether Nepal is underperforming, matching, or outperforming its peer groups.

In [27]:
query = """
SELECT DISTINCT region
FROM findex_clean
WHERE country_code = 'NPL'
ORDER BY region
"""
pd.read_sql(query,conn)

,region
0,South Asia (excluding high income)


In [28]:
query = """
SELECT DISTINCT income_group
FROM findex_clean
WHERE country_code = 'NPL'
ORDER BY income_group
"""
pd.read_sql(query, conn)

,income_group
0,Lower middle income


In [30]:
query = """
SELECT
    year,
    country,
    entity_type,
    region,
    income_group,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE "group" = 'all'
    AND "group2" = 'all'
    AND (
        country_code = 'NPL'
        OR country IN ('South Asia')
    )
ORDER BY year, country
"""
pd.read_sql(query, conn)

,year,country,entity_type,region,income_group,account_ownership_pct,digital_access_pct
0,2011,Nepal,Country,South Asia (excluding high income),Lower middle income,25.31,NaN
1,2011,South Asia,Aggregate,None,None,32.20,NaN
2,2014,Nepal,Country,South Asia (excluding high income),Lower middle income,33.80,NaN
3,2014,South Asia,Aggregate,None,None,46.32,NaN
4,2017,Nepal,Country,South Asia (excluding high income),Lower middle income,45.39,NaN
5,2017,South Asia,Aggregate,None,None,69.41,NaN
6,2021,Nepal,Country,South Asia (excluding high income),Lower middle income,54.00,NaN
7,2021,South Asia,Aggregate,None,None,68.00,NaN
8,2024,Nepal,Country,South Asia (excluding high income),Lower middle income,59.99,17.62
9,2024,South Asia,Aggregate,None,None,77.57,28.93


In [31]:
query = """
SELECT
    year,
    country,
    entity_type,
    income_group,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE "group" = 'all'
    AND "group2" = 'all'
    AND (
        country_code = 'NPL'
        OR income_group = 'Lower middle income'
    )
ORDER BY year, entity_type, country
LIMIT 30
"""
pd.read_sql(query, conn)

,year,country,entity_type,income_group,account_ownership_pct,digital_access_pct
0,2011,Algeria,Country,Lower middle income,33.29,None
1,2011,Angola,Country,Lower middle income,39.20,None
2,2011,Bangladesh,Country,Lower middle income,31.74,None
3,2011,Benin,Country,Lower middle income,10.46,None
4,2011,Bolivia,Country,Lower middle income,28.03,None
5,2011,Cambodia,Country,Lower middle income,3.66,None
6,2011,Cameroon,Country,Lower middle income,14.81,None
7,2011,Comoros,Country,Lower middle income,21.69,None
8,2011,"Congo, Rep.",Country,Lower middle income,10.05,None
9,2011,Djibouti,Country,Lower middle income,12.27,None


In [32]:
query = """
SELECT
    year,
    country,
    country_code,
    entity_type,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE "group" = 'all'
    AND "group2" = 'all'
    AND (
        country_code = 'NPL'
        OR country_code = 'SAS'
    )
ORDER BY year, country_code
"""
pd.read_sql(query, conn)

,year,country,country_code,entity_type,account_ownership_pct,digital_access_pct
0,2011,Nepal,NPL,Country,25.31,NaN
1,2011,South Asia,SAS,Aggregate,32.20,NaN
2,2014,Nepal,NPL,Country,33.80,NaN
3,2014,South Asia,SAS,Aggregate,46.32,NaN
4,2017,Nepal,NPL,Country,45.39,NaN
5,2017,South Asia,SAS,Aggregate,69.41,NaN
6,2021,Nepal,NPL,Country,54.00,NaN
7,2021,South Asia,SAS,Aggregate,68.00,NaN
8,2024,Nepal,NPL,Country,59.99,17.62
9,2024,South Asia,SAS,Aggregate,77.57,28.93


In [33]:
query = """
SELECT
    year,
    country,
    country_code,
    entity_type,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE "group" = 'all'
    AND "group2" = 'all'
    AND country_code IN ('NPL', 'SAS', 'WLD')
ORDER BY year, country_code
"""
pd.read_sql(query, conn)

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


,year,country,country_code,entity_type,account_ownership_pct,digital_access_pct
0,2011,Nepal,NPL,Country,25.31,NaN
1,2011,South Asia,SAS,Aggregate,32.20,NaN
2,2011,world,WLD,Aggregate,50.61,NaN
3,2014,Nepal,NPL,Country,33.80,NaN
4,2014,South Asia,SAS,Aggregate,46.32,NaN
5,2014,world,WLD,Aggregate,61.67,NaN
6,2017,Nepal,NPL,Country,45.39,NaN
7,2017,South Asia,SAS,Aggregate,69.41,NaN
8,2017,world,WLD,Aggregate,69.32,NaN
9,2021,Nepal,NPL,Country,54.00,NaN


In [34]:
query = """
SELECT
    year,
    country,
    country_code,
    entity_type,
    ROUND(account_ownership * 100, 2) AS account_ownership_pct,
    ROUND(digital_access * 100, 2) AS digital_access_pct
FROM findex_clean
WHERE "group" = 'all'
    AND "group2" = 'all'
    AND country_code IN ('NPL', 'LMC')
ORDER BY year, country_code
"""
pd.read_sql(query, conn)

,year,country,country_code,entity_type,account_ownership_pct,digital_access_pct
0,2011,Lower middle income,LMC,Aggregate,30.45,NaN
1,2011,Nepal,NPL,Country,25.31,NaN
2,2014,Lower middle income,LMC,Aggregate,43.64,NaN
3,2014,Nepal,NPL,Country,33.80,NaN
4,2017,Lower middle income,LMC,Aggregate,60.01,NaN
5,2017,Nepal,NPL,Country,45.39,NaN
6,2021,Lower middle income,LMC,Aggregate,62.11,NaN
7,2021,Nepal,NPL,Country,54.00,NaN
8,2024,Lower middle income,LMC,Aggregate,70.39,36.82
9,2024,Nepal,NPL,Country,59.99,17.62


## Benchmark Interpretation

Use these benchmark queries to compare Nepal with:
- South Asia aggregate (`SAS`)
- World aggregate (`WLD`)
- Lower middle income aggregate (`LMC`)

Focus on whether Nepal is above or below each benchmark for:
- account ownership
- digital access